In [ ]:
import time
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

# Fix seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Paths & toggles
TRAIN_CSV = "/content/sample_data/UNSW_NB15.csv"
LABEL_COL = "label"
SCALE_METHOD = "minmax"
RANDOM_STATE = SEED

# Utility functions
def safe_read_csv(path):
    if path is None:
        raise FileNotFoundError("CSV path is None. Set TRAIN_CSV in Cell 0.")
    return pd.read_csv(path)

def print_shape(df, name="df"):
    print(f"{name}.shape = {df.shape}")

# Show versions
import sklearn, tensorflow
print("numpy", np.__version__, "pandas", pd.__version__, "sklearn", sklearn.__version__, "tf", tensorflow.__version__)


numpy 2.0.2 pandas 2.2.2 sklearn 1.6.1 tf 2.19.0


In [ ]:

print("Loading & Preprocessing")
def make_binary(df):
    df = df.copy()
    df["label"] = df["Label"].apply(lambda x: "Normal" if x == "Normal" else "Attack")
    return df

# load train
df_train = safe_read_csv(TRAIN_CSV)
print_shape(df_train, "df_train (raw)")

# df_train = make_binary(df_train)
# df_train = df_train.drop(columns=["Label"])
# print(df_train.head())

df_train.columns = df_train.columns.str.strip()

useless_columns = [
    'id', 'attack_cat', 'srcip', 'dstip',
    # 'sport', 'dsport',
    'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd',
    'is_sm_ips_ports', 'response_body_len', 'trans_depth'
]

def drop_useless(df):
    cols = [c for c in useless_columns if c in df.columns]
    if cols:
        df = df.drop(columns=cols, errors='ignore')
    const_cols = [c for c in df.columns if df[c].nunique() <= 1]
    if const_cols:
        df = df.drop(columns=const_cols, errors='ignore')
    df = df.loc[:, ~df.T.duplicated()]
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df

df_train = drop_useless(df_train)
print_shape(df_train, "df_train (cleaned)")

# Separate X/y for train
if LABEL_COL not in df_train.columns:
    raise KeyError(f"Label column '{LABEL_COL}' not found in training CSV")
y_train_raw = df_train[LABEL_COL].astype(str)
X_train_df = df_train.drop(columns=[LABEL_COL])
print(X_train_df.head())
print_shape(X_train_df, "X_train")

# Encode categorical columns
def one_hot_small_cardinality(df, max_unique=20, exclude=[]):
    cat_cols = [c for c in df.columns if df[c].dtype == 'object' and c not in exclude]
    low_card = [c for c in cat_cols if df[c].nunique() <= max_unique]
    if low_card:
        df = pd.get_dummies(df, columns=low_card, prefix=low_card)
    for c in df.columns:
        if df[c].dtype == 'object':
            df[c] = LabelEncoder().fit_transform(df[c].astype(str))
    return df

X_train_df = one_hot_small_cardinality(X_train_df)

print("Final feature count:", X_train_df.shape[1])


Loading & Preprocessing
df_train (raw).shape = (175341, 36)
df_train (cleaned).shape = (175341, 29)
        dur proto service state  spkts  dpkts  sbytes  dbytes       rate  \
0  0.121478   tcp       -   FIN      6      4     258     172  74.087490   
1  0.649902   tcp       -   FIN     14     38     734   42014  78.473370   
2  1.623129   tcp       -   FIN      8     16     364   13186  14.170161   
3  1.681642   tcp     ftp   FIN     12     12     628     770  13.677108   
4  0.449454   tcp       -   FIN     10      6     534     268  33.373825   

        sload  ...       stcpb       dtcpb  dwin    tcprtt    synack  \
0  14158.9420  ...   621772692  2202533631   255  0.000000  0.000000   
1   8395.1120  ...  1417884146  3077387971   255  0.000000  0.000000   
2   1572.2719  ...  2116150707  2963114973   255  0.111897  0.061458   
3   2740.1790  ...  1107119177  1047442890   255  0.000000  0.000000   
4   8561.4990  ...  2436137549  1977154190   255  0.128381  0.071147   

     ackda

In [ ]:

print("Scaling & split")
if SCALE_METHOD == "minmax":
    scaler = MinMaxScaler()
else:
    scaler = StandardScaler()


X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_df), columns=X_train_df.columns, index=X_train_df.index)
X_test_scaled = None

# Encode labels for model training
le_target = LabelEncoder()
y_train_enc = le_target.fit_transform(y_train_raw)
class_names = list(le_target.classes_)
print("Classes:", class_names)


X_train, X_test, Y_train, y_test = train_test_split(
    X_train_scaled.values, y_train_enc, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_enc
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)


Scaling & split
Classes: ['0', '1']
X_train: (140272, 48) X_test: (35069, 48)


In [ ]:

print("Feature scoring: Information Gain + Fisher score (Weighted RMS)")
from sklearn.feature_selection import mutual_info_classif

# Measure training time
start_time = time.time()

# Information Gain
mi = mutual_info_classif(X_train_scaled.values, y_train_enc, random_state=RANDOM_STATE)
mi_norm = (mi - mi.min()) / (mi.max() - mi.min() + 1e-12)

# Fisher Score
def fisher_score(X, y):
    X = np.asarray(X, dtype=np.float64)
    n_samples, n_features = X.shape
    classes = np.unique(y)
    overall_mean = np.mean(X, axis=0)
    scores = np.zeros(n_features, dtype=np.float64)
    for c in classes:
        idx = np.where(y == c)[0]
        if len(idx) == 0:
            continue
        Xc = X[idx]
        nc = len(idx)
        mean_c = np.mean(Xc, axis=0)
        var_c = np.var(Xc, axis=0)
        var_c = np.where(var_c == 0, 1e-9, var_c)
        scores += nc * (mean_c - overall_mean) ** 2 / var_c
    return scores

f_scores = fisher_score(X_train_scaled.values, y_train_enc)
f_norm = (f_scores - f_scores.min()) / (f_scores.max() - f_scores.min() + 1e-12)

# Weighted RMS Combination
alpha = 0.6
beta  = 0.4

weighted_rms = np.sqrt((alpha * mi_norm**2 + beta * f_norm**2))

# Ranked DataFrame
rank_df = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "info_gain_norm": mi_norm,
    "fisher_norm": f_norm,
    "weighted_rms": weighted_rms
}).sort_values("weighted_rms", ascending=False).reset_index(drop=True)
print("Full Feature Ranking by Weighted RMS")
print(rank_df.to_string(index=False))


Feature scoring: Information Gain + Fisher score (Weighted RMS)
Full Feature Ranking by Weighted RMS
         feature  info_gain_norm  fisher_norm  weighted_rms
          sbytes        1.000000 6.075819e-04  7.745968e-01
    service_dhcp        0.004036 1.000000e+00  6.324633e-01
          dbytes        0.793755 6.631952e-05  6.148396e-01
            rate        0.755969 5.331888e-03  5.855808e-01
           sload        0.741710 4.877768e-04  5.745264e-01
             dur        0.725375 2.096254e-05  5.618727e-01
           smean        0.715461 1.328105e-06  5.541937e-01
           dmean        0.670204 1.277306e-03  5.191381e-01
           dload        0.599901 2.679444e-01  4.946173e-01
          dinpkt        0.636044 6.821815e-06  4.926773e-01
           dpkts        0.581662 1.613671e-04  4.505536e-01
          sinpkt        0.528588 4.668191e-03  4.094528e-01
          tcprtt        0.506706 6.577582e-05  3.924926e-01
          synack        0.503608 3.266441e-05  3.900933e-01

In [ ]:

print("WOA search for optimal k (top-k feature selection)")

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
import math

def top_k_avg_rms(rank_df, k):
    # Handle edge cases
    if k <= 0:
        raise ValueError("K must be a positive integer.")
    if k > len(rank_df):
        raise ValueError(f"K ({k}) cannot be greater than number of rows in rank_df ({len(rank_df)}).")

    # Select top K rows based on weighted_rms
    top_k = rank_df.nlargest(k, "weighted_rms")["weighted_rms"]

    # Return the mean
    return top_k.mean()


def whale_optimize_k(
    ranking_features,
    X_train, y_train,
    X_val, y_val,
    n_whales=20,
    max_iter=30,
    penalty=0.005,
    beta=0.005,
    subsample_ratio=0.5,
    rng_seed=SEED
):
    # run WOA

    rng = np.random.default_rng(rng_seed)
    n_features = len(ranking_features)


    subsample_size = max(100, int(subsample_ratio * X_train.shape[0]))
    subsample_idx = rng.choice(X_train.shape[0], size=subsample_size, replace=False)
    X_sub = X_train[subsample_idx]
    y_sub = y_train[subsample_idx]


    fitness_cache = {}

    def eval_k(k_val):
        # Evaluate a given k using a lightweight RandomForest classifier.
        k_int = int(np.clip(round(k_val), 1, n_features))
        if k_int in fitness_cache:
            return fitness_cache[k_int]

        selected_cols = ranking_features[:k_int]
        col_idxs = [list(X_train_df.columns).index(c) for c in selected_cols]

        clf = RandomForestClassifier(n_estimators=30, max_depth=10,
                                     random_state=RANDOM_STATE, n_jobs=-1)
        clf.fit(X_sub[:, col_idxs], y_sub)
        preds = clf.predict(X_val[:, col_idxs])
        acc = accuracy_score(y_val, preds)

        avg_HG = top_k_avg_rms(rank_df,k_int)

        fitness = (1 - acc) + penalty * (k_int / n_features) + beta*(avg_HG)
        fitness_cache[k_int] = (fitness, acc, k_int)
        return fitness, acc, k_int

    # Initialize
    k_pop = rng.integers(1, n_features + 1, size=n_whales).astype(float)
    print(k_pop)

    # Evaluate initial population
    fitness = np.zeros(n_whales)
    acc_pop = np.zeros(n_whales)
    for i in range(n_whales):
        f, a, _ = eval_k(k_pop[i])
        fitness[i] = f
        acc_pop[i] = a

    # Determine the best whale initially
    best_idx = np.argmin(fitness)
    best_k = int(round(k_pop[best_idx]))
    best_fit = fitness[best_idx]
    best_acc = acc_pop[best_idx]

    # WOA loop
    b = 1.0
    for t in range(1, max_iter + 1):
        a = 2 * (1 - t / max_iter)

        for i in range(n_whales):
            r1, r2 = rng.random(), rng.random()
            A = 2 * a * r1 - a
            C = 2 * r1
            p = rng.random()
            l = rng.uniform(-1, 1)

            k_i = k_pop[i]
            if abs(A) < 1:
              if p < 0.5:
                  D = abs(C * best_k - k_i)
                  new_k = best_k - A * D
              else:
                  D = abs(C * best_k - k_i)
                  new_k = D * math.exp(b * l) * math.cos(2 * math.pi * l) + best_k
            else:
                j = rng.integers(0, n_whales)
                while j == i:
                    j = rng.integers(0, n_whales)
                Xj = k_pop[j]
                D = abs(C * Xj - k_i)
                new_k = Xj - A * D


            new_k = float(np.clip(new_k, 1, n_features))
            f_new, acc_new, k_new_int = eval_k(new_k)

            k_pop[i] = new_k
            fitness[i] = f_new
            acc_pop[i] = acc_new


            if f_new < best_fit or (math.isclose(f_new, best_fit) and acc_new > best_acc):
                best_fit = f_new
                best_acc = acc_new
                best_k = k_new_int


        if t % max(1, max_iter // 5) == 0 or t == 1 or t == max_iter:
            print(f"WOA Iter {t}/{max_iter} — best_k={best_k}, acc={best_acc:.4f}, fit={best_fit:.4f}")

    return best_k, best_acc, best_fit



ranking_list = rank_df['feature'].tolist()

best_k, best_acc, best_fit = whale_optimize_k(
    ranking_list,
    X_train, Y_train,
    X_test, y_test,
    n_whales=15, max_iter=20, penalty=0.01, beta=0.001, subsample_ratio=0.7, rng_seed=SEED
)

print("\n WOA finished (optimized).")
print(f"Best number of features found: {best_k}")
print(f"Approx. validation accuracy (light RF): {best_acc:.4f}")


WOA search for optimal k (top-k feature selection)
[22.  7. 27.  1. 32. 19. 32.  7. 24.  9. 27.  3. 26. 21.  7. 36. 41. 27.
 25. 18. 14.  1. 24. 25.  2. 16. 27. 12. 29. 22. 45.  9.  2. 16. 39.]
WOA Iter 1/50 — best_k=27, acc=0.9417, fit=0.0643
WOA Iter 10/50 — best_k=27, acc=0.9417, fit=0.0643
WOA Iter 20/50 — best_k=27, acc=0.9417, fit=0.0643
WOA Iter 30/50 — best_k=27, acc=0.9417, fit=0.0643
WOA Iter 40/50 — best_k=27, acc=0.9417, fit=0.0643
WOA Iter 50/50 — best_k=27, acc=0.9417, fit=0.0643

 WOA finished (optimized).
Best number of features found: 27
Approx. validation accuracy (light RF): 0.9417


In [ ]:

import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("--- Final evaluation with top-k features ---")
top_k_features = ranking_list[:best_k]
sel_idxs = [list(X_train_scaled.columns).index(c) for c in top_k_features]

X_full = X_train_scaled.values
y_full = y_train_enc

X_test_for_eval = X_test[:, sel_idxs]
y_test_for_eval = y_test



final_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_rf.fit(X_full[:, sel_idxs], y_full)

end_time = time.time()
training_time = end_time - start_time

y_pred = final_rf.predict(X_test_for_eval)

print("Final RF accuracy on hold-out:", accuracy_score(y_test_for_eval, y_pred))
print(classification_report(y_test_for_eval, y_pred, target_names=class_names))
print(f"Training time for best_k = {best_k}: {training_time:.4f} seconds")


--- Final evaluation with top-k features ---
Final RF accuracy on hold-out: 0.9926430750805555
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     11200
           1       0.99      1.00      0.99     23869

    accuracy                           0.99     35069
   macro avg       0.99      0.99      0.99     35069
weighted avg       0.99      0.99      0.99     35069

Training time for best_k = 27: 304.4275 seconds


In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Final evaluation with ALL features (no feature selection)")

# Use every feature
X_full = X_train_scaled.values
y_full = y_train_enc

X_test_for_eval = X_test  # full feature test set
y_test_for_eval = y_test

# Measure training time
start_time = time.time()

final_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_rf.fit(X_full, y_full)

end_time = time.time()
training_time = end_time - start_time

# Predictions
y_pred = final_rf.predict(X_test_for_eval)

print("Final RF accuracy on hold-out:", accuracy_score(y_test_for_eval, y_pred))
print(classification_report(y_test_for_eval, y_pred, target_names=class_names))
# print(f"Training time with ALL features: {training_time:.4f} seconds")


--- Final evaluation with ALL features (no feature selection) ---
Final RF accuracy on hold-out: 1.0
              precision    recall  f1-score   support

      Attack       1.00      1.00      1.00     44075
      Normal       1.00      1.00      1.00     10948

    accuracy                           1.00     55023
   macro avg       1.00      1.00      1.00     55023
weighted avg       1.00      1.00      1.00     55023

Training time with ALL features: 91.3448 seconds


In [ ]:
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(X_train, y_train, X_test, y_test, label):
    start = time.time()

    clf = RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    train_time = time.time() - start
    y_pred = clf.predict(X_test)

    return {
        "Model": label,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='macro'),
        "Recall": recall_score(y_test, y_pred, average='macro'),
        "F1": f1_score(y_test, y_pred, average='macro'),
        "Training Time (sec)": train_time,
        "Num Features": X_train.shape[1]
    }


X_full_train = X_train_scaled.values
y_full_train = y_train_enc

X_full_test = X_test
y_full_test = y_test

results_all = evaluate_model(
    X_full_train, y_full_train,
    X_full_test, y_full_test,
    label="All Features"
)

top_k_features = ranking_list[:best_k]
sel_idxs = [list(X_train_scaled.columns).index(c) for c in top_k_features]

X_topk_train = X_train_scaled.values[:, sel_idxs]
X_topk_test = X_test[:, sel_idxs]

results_topk = evaluate_model(
    X_topk_train, y_full_train,
    X_topk_test, y_full_test,
    label=f"Top-{best_k} Features"
)

comparison_df = pd.DataFrame([results_all, results_topk])

print("\n=== Model Comparison ===")
print(comparison_df.to_string(index=False))



=== Model Comparison ===
          Model  Accuracy  Precision   Recall       F1  Training Time (sec)  Num Features
   All Features  0.992672   0.992848 0.990280 0.991550            66.020627            48
Top-27 Features  0.992643   0.992827 0.990236 0.991517            85.745802            27


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("=== Logistic Regression ===")
lr = LogisticRegression(max_iter=2000, n_jobs=-1)
lr.fit(X_train_sel, y_train_sel)

y_pred_lr = lr.predict(X_test_sel)

print("Accuracy:", accuracy_score(y_test_sel, y_pred_lr))
print(classification_report(y_test_sel, y_pred_lr, target_names=class_names))


In [ ]:
from sklearn.svm import SVC

print("=== SVM (RBF Kernel) ===")
svm = SVC(kernel='rbf', probability=True)
svm.fit(X_train_sel, y_train_sel)

y_pred_svm = svm.predict(X_test_sel)

print("Accuracy:", accuracy_score(y_test_sel, y_pred_svm))
print(classification_report(y_test_sel, y_pred_svm, target_names=class_names))


In [ ]:
from sklearn.naive_bayes import GaussianNB

print("=== Naive Bayes ===")
nb = GaussianNB()
nb.fit(X_train_sel, y_train_sel)

y_pred_nb = nb.predict(X_test_sel)

print("Accuracy:", accuracy_score(y_test_sel, y_pred_nb))
print(classification_report(y_test_sel, y_pred_nb, target_names=class_names))


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

print("=== KNN (k=5) ===")
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sel, y_train_sel)

y_pred_knn = knn.predict(X_test_sel)

print("Accuracy:", accuracy_score(y_test_sel, y_pred_knn))
print(classification_report(y_test_sel, y_pred_knn, target_names=class_names))


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
import pandas as pd
acc = accuracy_score(y_test_for_eval, y_pred)
print("Accuracy:", acc)
cm = confusion_matrix(y_test_for_eval, y_pred)
print("\nConfusion Matrix (Counts):\n", cm)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
print("\nConfusion Matrix (Normalized):\n", cm_norm)

print("\nClassification Report:")
print(classification_report(y_test_for_eval, y_pred, target_names=class_names))

df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
print("\nConfusion Matrix Table:")
print(df_cm)

df_cm_norm = pd.DataFrame(cm_norm, index=class_names, columns=class_names)
print("\nNormalized Confusion Matrix Table:")
print(df_cm_norm)


Accuracy: 0.9926430750805555

Confusion Matrix (Counts):
 [[11016   184]
 [   74 23795]]

Confusion Matrix (Normalized):
 [[0.98357143 0.01642857]
 [0.00310026 0.99689974]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     11200
           1       0.99      1.00      0.99     23869

    accuracy                           0.99     35069
   macro avg       0.99      0.99      0.99     35069
weighted avg       0.99      0.99      0.99     35069


Confusion Matrix Table:
       0      1
0  11016    184
1     74  23795

Normalized Confusion Matrix Table:
          0         1
0  0.983571  0.016429
1  0.003100  0.996900
